<a href="https://colab.research.google.com/github/janani-pb/amazonprime_content_analysis/blob/main/AmazonPrime_content__analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy matplotlib seaborn sqlite3

ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set(style="whitegrid")

In [3]:
prime_df = pd.read_csv("amazon_prime_titles.csv")
imdb_df = pd.read_csv("imdb_dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'imdb_dataset.csv'

In [ ]:
print(prime_df.head())
print(prime_df.info())


In [ ]:
print(imdb_df.head())
print(imdb_df.info())

In [ ]:
print(prime_df.columns)

In [ ]:
prime_df.rename(columns={'listed_in': 'genre'}, inplace=True)

In [ ]:
prime_df = prime_df.dropna(subset=['genre', 'country'])



In [ ]:
imdb_df = imdb_df.dropna(subset=['Genre', 'IMDB_Rating'])

In [ ]:
imdb_df.rename(columns={
    'Series_Title': 'title',
    'Genre': 'genre',
    'IMDB_Rating': 'rating'
}, inplace=True)

In [ ]:
prime_df['title'] = prime_df['title'].str.lower()

In [ ]:
imdb_df['title'] = imdb_df['title'].str.lower()

In [ ]:
merged_df = pd.merge(prime_df, imdb_df, on='title', how='inner')
print(merged_df.head(5))

In [ ]:
print(merged_df.columns)

In [ ]:
merged_df['genre'] = merged_df['genre_x']

In [ ]:
merged_df['genre'] = merged_df['genre_x'].fillna(merged_df['genre_y'])

In [ ]:
print(prime_df.columns)

In [ ]:
print(imdb_df.columns)

In [ ]:
merged_df['genre'] = merged_df['genre'].str.split(',')
merged_df = merged_df.explode('genre')
merged_df['genre'] = merged_df['genre'].str.strip()

In [ ]:
genre_counts = merged_df['genre'].value_counts()

print(genre_counts.head(10))

In [ ]:
plt.figure(figsize=(12,6))
genre_counts.head(10).plot(kind='bar')
plt.title("Top 10 Genres on Amazon Prime")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
country_counts = merged_df['country'].value_counts()

print(country_counts.head(10))

In [ ]:
plt.figure(figsize=(12,6))
country_counts.head(10).plot(kind='bar', color='orange')
plt.title("Top 10 Content Producing Countries")
plt.xlabel("Country")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.show()

In [ ]:
pivot_table = pd.pivot_table(
    merged_df,
    values='title',
    index='country',
    columns='genre',
    aggfunc='count',
    fill_value=0
)

plt.figure(figsize=(12,8))
sns.heatmap(pivot_table.head(10), cmap='coolwarm')
plt.title("Genre vs Country Heatmap")
plt.show()

In [ ]:
print(merged_df.columns)

In [ ]:
merged_df.rename(columns={'IMDB_Rating': 'rating'}, inplace=True)

In [ ]:
merged_df['rating'] = merged_df['rating_x'].fillna(merged_df['rating_y'])

In [ ]:
top_genres = merged_df.groupby('genre')['rating_y'].mean().sort_values(ascending=False)

print(top_genres.head(10))

In [ ]:
plt.figure(figsize=(10,5))
top_genres.head(10).plot(kind='bar', color='green')
plt.title("Top Rated Genres (IMDb Ratings)")
plt.ylabel("Average Rating")
plt.xticks(rotation=45)
plt.show()

In [ ]:
merged_df.rename(columns={
    'director': 'prime_director',
    'Director': 'imdb_director'
}, inplace=True)

In [ ]:
conn = sqlite3.connect("streaming_analysis.db")
merged_df.to_sql("prime_imdb", conn, if_exists='replace', index=False)

In [ ]:
# Rename important columns
merged_df.rename(columns={
    'Director': 'imdb_director',
    'rating_y': 'rating'
}, inplace=True)

# Remove unnecessary duplicates
merged_df.drop(columns=['genre_x', 'genre_y', 'rating_x'], inplace=True, errors='ignore')

In [ ]:
print(merged_df.columns)

In [ ]:
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

In [ ]:
print(merged_df.columns)

In [ ]:
import sqlite3

conn = sqlite3.connect("streaming_analysis.db")
merged_df.to_sql("prime_imdb", conn, if_exists='replace', index=False)

In [ ]:
conn = sqlite3.connect("streaming_analysis.db")
merged_df.to_sql("prime_imdb", conn, if_exists='replace', index=False)

In [ ]:
query1 = """
SELECT genre, COUNT(*) as count
FROM prime_imdb
GROUP BY genre
ORDER BY count DESC
LIMIT 10;
"""
print(pd.read_sql(query1, conn))

In [ ]:
query2 = """
SELECT country, COUNT(*) as total
FROM prime_imdb
GROUP BY country
ORDER BY total DESC
LIMIT 10;
"""
print(pd.read_sql(query2, conn))

In [ ]:
query3 = """
SELECT genre, AVG(rating) as avg_rating
FROM prime_imdb
GROUP BY genre
ORDER BY avg_rating DESC
LIMIT 10;
"""
print(pd.read_sql(query3, conn))